In [1]:
import numpy as np
import pandas as pd
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

from scipy.signal import butter, filtfilt, savgol_filter

In [2]:
def load_ucihar_signals(folder_path):
    signal_files = [
        "body_acc_x_", "body_acc_y_", "body_acc_z_",
        "body_gyro_x_", "body_gyro_y_", "body_gyro_z_",
        "total_acc_x_", "total_acc_y_", "total_acc_z_"
    ]

    signals = []

    for signal in signal_files:
        file_path = os.path.join(folder_path, signal + folder_path.split('/')[-2] + ".txt")
        data = np.loadtxt(file_path)
        signals.append(data)

    # Stack → (samples, timesteps, features)
    X = np.stack(signals, axis=2)
    return X

In [3]:
def load_labels(file_path):
    return np.loadtxt(file_path).astype(int)

In [4]:
base_path = "UCI HAR Dataset/UCI HAR Dataset/"

X_train = load_ucihar_signals(os.path.join(base_path, "train/Inertial Signals"))
y_train = load_labels(os.path.join(base_path, "train/y_train.txt"))

X_test = load_ucihar_signals(os.path.join(base_path, "test/Inertial Signals"))
y_test = load_labels(os.path.join(base_path, "test/y_test.txt"))

# Combine
X = np.concatenate([X_train, X_test], axis=0)
y = np.concatenate([y_train, y_test], axis=0)

print(X.shape)  # (10299, 128, 9)

(10299, 128, 9)


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [6]:
scaler = StandardScaler()
X = scaler.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

In [7]:
def downsample_naive(X, factor=2):
    return X[:, ::factor, :]

def downsample_mean(X, factor=2):
    N, T, F = X.shape
    T_new = T // factor
    X = X[:, :T_new*factor, :]
    return X.reshape(N, T_new, factor, F).mean(axis=2)

In [8]:
def butter_filter(X, cutoff=5, fs=50, order=3):
    b, a = butter(order, cutoff / (0.5 * fs), btype='low')
    X_filtered = np.zeros_like(X)

    for i in range(X.shape[0]):
        for j in range(X.shape[2]):
            X_filtered[i, :, j] = filtfilt(b, a, X[i, :, j])

    return X_filtered


def savgol_filtering(X, window=7, poly=2):
    X_filtered = np.zeros_like(X)

    for i in range(X.shape[0]):
        for j in range(X.shape[2]):
            X_filtered[i, :, j] = savgol_filter(X[i, :, j], window, poly)

    return X_filtered

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten, MaxPooling1D
from tensorflow.keras import Input

def build_tiny_cnn(input_shape, num_classes):

    model = Sequential([
        Input(shape=input_shape),
        Conv1D(16, 3, activation='relu', input_shape=input_shape),
        MaxPooling1D(2),

        Conv1D(32, 3, activation='relu'),
        MaxPooling1D(2),

        Conv1D(32, 3, activation='relu'),

        Flatten(),
        Dense(32, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def get_models():
    return {
        "LogisticRegression": LogisticRegression(max_iter=200),
        "DecisionTree": DecisionTreeClassifier(max_depth=15, min_samples_split=10),
        "TinyCNN": "cnn"
    }

In [11]:
import time
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.base import clone

def train_and_evaluate(X_train, y_train, X_test, y_test, model):

    input_shape = X_train.shape

    # =====================================================
    # CNN CASE
    # =====================================================
    if isinstance(model, str) and model == "cnn":

        if len(X_train.shape) != 3:
            raise ValueError("CNN requires (samples, timesteps, features)")

        num_classes = len(np.unique(y_train))

        cnn = build_tiny_cnn(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            num_classes=num_classes
        )

        start_train = time.time()
        cnn.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)
        train_time = time.time() - start_train

        start_inf = time.time()
        y_pred = np.argmax(cnn.predict(X_test, verbose=0), axis=1)
        inference_time = time.time() - start_inf

    # =====================================================
    # SKLEARN MODELS
    # =====================================================
    else:

        model_copy = clone(model)

        # SAFE FLATTENING (UCI-HAR compatible)
        X_train_flat = X_train.reshape(X_train.shape[0], -1)
        X_test_flat = X_test.reshape(X_test.shape[0], -1)

        start_train = time.time()
        model_copy.fit(X_train_flat, y_train)
        train_time = time.time() - start_train

        start_inf = time.time()
        y_pred = model_copy.predict(X_test_flat)
        inference_time = time.time() - start_inf

    # =====================================================
    # METRICS
    # =====================================================
    acc = accuracy_score(y_test, y_pred)
    inf_per_sample = inference_time / len(X_test)

    return {
        "accuracy": acc,
        "train_time": train_time,
        "inference_time": inference_time,
        "inference_time_per_sample": inf_per_sample,
        "input_shape": input_shape
    }

In [12]:
# def train_and_evaluate(X_train, y_train, X_test, y_test):

#     input_shape = X_train.shape

#     X_train_flat = X_train.reshape(X_train.shape[0], -1)
#     X_test_flat = X_test.reshape(X_test.shape[0], -1)

#     model = RandomForestClassifier(n_estimators=100, random_state=42)

#     start_train = time.time()
#     model.fit(X_train_flat, y_train)
#     train_time = time.time() - start_train

#     start_inf = time.time()
#     y_pred = model.predict(X_test_flat)
#     inference_time = time.time() - start_inf

#     acc = accuracy_score(y_test, y_pred)

#     return {
#         "accuracy": acc,
#         "train_time": train_time,
#         "inference_time": inference_time,
#         "inference_time_per_sample": inference_time / len(X_test),
#         "input_shape": input_shape
#     }

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
def run_all_experiments(X, y):

    results = []
    models = get_models()

    from sklearn.model_selection import train_test_split
    from sklearn.base import clone

    # Precompute
    X_naive = downsample_naive(X)
    X_mean = downsample_mean(X)
    X_butter = butter_filter(X)
    X_savgol = savgol_filtering(X)

    pipelines = [
        ("Raw", X, "No Processing"),

        ("Naive Downsample", X_naive, "Downsampling Only"),
        ("Mean Downsample", X_mean, "Downsampling Only"),

        ("Butterworth", X_butter, "Filtering Only"),
        ("Savitzky-Golay", X_savgol, "Filtering Only"),

        ("Naive + Butterworth", butter_filter(X_naive), "Downsampling → Filtering"),
        ("Mean + Butterworth", butter_filter(X_mean), "Downsampling → Filtering"),
        ("Naive + Savgol", savgol_filtering(X_naive), "Downsampling → Filtering"),
        ("Mean + Savgol", savgol_filtering(X_mean), "Downsampling → Filtering"),

        ("Butterworth → Naive", downsample_naive(X_butter), "Filtering → Downsampling"),
        ("Butterworth → Mean", downsample_mean(X_butter), "Filtering → Downsampling"),
        ("Savgol → Naive", downsample_naive(X_savgol), "Filtering → Downsampling"),
        ("Savgol → Mean", downsample_mean(X_savgol), "Filtering → Downsampling"),
    ]

    for model_name, model in models.items():

        print(f"\nRunning for model: {model_name}")

        for name, X_proc, pipeline_type in pipelines:

            X_train, X_test, y_train, y_test = train_test_split(
                X_proc, y, test_size=0.2, random_state=42
            )

            if model == "cnn":
                metrics = train_and_evaluate(X_train, y_train, X_test, y_test, model)
            else:
                from sklearn.base import clone
                model_copy = clone(model)
                metrics = train_and_evaluate(X_train, y_train, X_test, y_test, model_copy)

            metrics["name"] = name
            metrics["pipeline_type"] = pipeline_type
            metrics["model"] = model_name

            results.append(metrics)

    return pd.DataFrame(results)

In [15]:
# def run_all_experiments(X, y):

#     results = []

#     def evaluate_pipeline(name, X_proc, pipeline_type):
#         X_train, X_test, y_train, y_test = train_test_split(
#             X_proc, y, test_size=0.2, random_state=42
#         )

#         metrics = train_and_evaluate(X_train, y_train, X_test, y_test)
#         metrics["name"] = name
#         metrics["pipeline_type"] = pipeline_type
#         results.append(metrics)

#     # Raw
#     evaluate_pipeline("Raw", X, "No Processing")

#     # Down only
#     evaluate_pipeline("Naive Downsample", downsample_naive(X), "Downsampling Only")
#     evaluate_pipeline("Mean Downsample", downsample_mean(X), "Downsampling Only")

#     # Filter only
#     evaluate_pipeline("Butterworth", butter_filter(X), "Filtering Only")
#     evaluate_pipeline("Savitzky-Golay", savgol_filtering(X), "Filtering Only")

#     # Down → Filter
#     evaluate_pipeline("Naive + Butterworth", butter_filter(downsample_naive(X)), "Downsampling → Filtering")
#     evaluate_pipeline("Naive + Savgol", savgol_filtering(downsample_naive(X)), "Downsampling → Filtering")
#     evaluate_pipeline("Mean + Butterworth", butter_filter(downsample_mean(X)), "Downsampling → Filtering")
#     evaluate_pipeline("Mean + Savgol", savgol_filtering(downsample_mean(X)), "Downsampling → Filtering")

#     # Filter → Down
#     evaluate_pipeline("Butterworth → Naive", downsample_naive(butter_filter(X)), "Filtering → Downsampling")
#     evaluate_pipeline("Butterworth → Mean", downsample_mean(butter_filter(X)), "Filtering → Downsampling")
#     evaluate_pipeline("Savgol → Naive", downsample_naive(savgol_filtering(X)), "Filtering → Downsampling")
#     evaluate_pipeline("Savgol → Mean", downsample_mean(savgol_filtering(X)), "Filtering → Downsampling")

#     return pd.DataFrame(results)

In [16]:
df_results = run_all_experiments(X, y)
# df_results["dataset"] = "WISDM"   # change accordingly


Running for model: LogisticRegression


C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/m


Running for model: DecisionTree

Running for model: TinyCNN


C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(sha

In [17]:
def compute_score(row):
    return (
        row["accuracy"]
        - 0.1 * row["inference_time"]
        - 0.000001 * row["train_time"]
    )

In [18]:
df_results["score"] = df_results.apply(compute_score, axis=1)

In [20]:
models = get_models()

In [23]:
models = df_results["model"].unique()

for m in models:

    print("\n" + "="*60)
    print(f"🔹 MODEL: {m}")
    print("="*60)

    df_model = df_results[df_results["model"] == m]

    # already sorted above, but ensure safety
    df_model = df_model.sort_values(by="score", ascending=False)

    # reorder columns EXACTLY like your target format
    df_model = df_model[
        ["model", "name", "pipeline_type",
         "accuracy", "train_time",
         "inference_time", "inference_time_per_sample",
         "input_shape", "score"]
    ]

    print(df_model.to_string(index=False))


🔹 MODEL: LogisticRegression
             model                name            pipeline_type  accuracy  train_time  inference_time  inference_time_per_sample    input_shape    score
LogisticRegression                 Raw            No Processing  0.611650    4.680891        0.004026               1.954222e-06 (8239, 128, 9) 0.611243
LogisticRegression      Savitzky-Golay           Filtering Only  0.611650    4.604291        0.005281               2.563694e-06 (8239, 128, 9) 0.611118
LogisticRegression      Savgol → Naive Filtering → Downsampling  0.609223    2.579819        0.002991               1.452154e-06  (8239, 64, 9) 0.608922
LogisticRegression     Mean Downsample        Downsampling Only  0.608738    2.627415        0.002000               9.708034e-07  (8239, 64, 9) 0.608535
LogisticRegression  Butterworth → Mean Filtering → Downsampling  0.608252    2.573138        0.000000               0.000000e+00  (8239, 64, 9) 0.608250
LogisticRegression         Butterworth           Filt